## NB02-Data transformation

### Load raw data

In [1]:
import json
import pandas as pd

In [2]:
with open("../data/raw/movies.json", "r") as f:
    all_pages = json.load(f)

with open("../data/raw/genres.json", "r") as f:
    genres_raw = json.load(f)

### Decision: flatten pages

In [3]:
all_movies = []
for page_data in all_pages:
    all_movies.extend(page_data["results"]) #`.extend()` adds all items from a list individually (unlike `.append()`, which would add the whole `results` list as one nested item). 

print(f"Total movie records collected: {len(all_movies)}")

Total movie records collected: 1000




`all_pages` is a list of 50 API responses — one dict per page, each holding
its own `results` list of 20 movies  To analyse movies as a single
table later, we need one flat list of movie records instead of 50 separate
page-dicts.



### Build the movies and genre DataFrames

In [4]:
genres_df = pd.DataFrame(genres_raw["genres"])
genres_df.head()

,id,name
0,28,Action
1,12,Adventure
2,16,Animation
3,35,Comedy
4,80,Crime


In [5]:
movies_df = pd.DataFrame(all_movies)
movies_df.head()

,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
0,False,/RMXG8myu1aGlNUsRjtxzmpdMK0.jpg,"[12, 28, 14]",1368337,The Odyssey,en,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...",1012.4511,/5rhTDKUhPYvpdQIijFIs5VoWsON.jpg,2026-07-15,False,False,8.000,1628
1,False,/flxau5Iu7bChQHsESqvGZ3FQRaI.jpg,"[878, 53]",1275779,Disclosure Day,en,Disclosure Day,A cybersecurity expert becomes a whistleblower...,544.2213,/AnJ8IQJI23hNpYXVNaythu061Ru.jpg,2026-06-10,False,False,7.392,1896
2,False,/piV2OnzTZCyGBP9JCjlHIgKGlfo.jpg,"[28, 14, 878]",454639,Masters of the Universe,en,Masters of the Universe,"After being separated for 15 years, the Sword ...",525.9230,/oRuyGUHdoaQxWP3SDfafGkStxTC.jpg,2026-06-03,False,False,7.338,1204
3,False,/c6BPbkO5Npt1OdwttAxCFo06wtH.jpg,"[10751, 14, 35, 12]",1108427,Moana,en,Moana,"Teenage Moana answers the Ocean's call and, fo...",463.6699,/zKVgiv5qHCvCLT4A2ymJi5QeXDH.jpg,2026-07-08,False,False,5.900,147
4,False,/wZJTidyAi53smx6x4LBFgfQolRU.jpg,"[28, 12, 878]",1081003,Supergirl,en,Supergirl,When an unexpected and ruthless adversary stri...,511.2893,/niSvU02l2BONH9ivubV6K1a5QiK.jpg,2026-06-24,False,False,6.600,766


### Deleting duplicated values

In [17]:
print("Duplicate movie ids:", movies_df["id"].duplicated().sum())
movies_df = movies_df.drop_duplicates(subset="id")

Duplicate movie ids: 10


### Filtering the data set

Analizing the `vote_count` and `popularity`. TMDB's `popularity` is a platform-calculated score reflecting current interest and activity (page views, recent votes, watchlist/favourite additions, release
recency) — not perceived quality.

In [9]:
print("Minimum vote_count:", movies_df["vote_count"].min(), "/  Mean vote_count:", movies_df["vote_count"].mean(), "/  Maximum vote_count:", movies_df["vote_count"].max())
print("Minimum popularity:", movies_df["popularity"].min(), "/  Mean popularity:", movies_df["popularity"].mean(), "/  Maximum popularity:", movies_df["popularity"].max())

Minimum vote_count: 0 /  Mean vote_count: 6724.048 /  Maximum vote_count: 40471
Minimum popularity: 9.049 /  Mean popularity: 29.962559799999998 /  Maximum popularity: 1012.4511


In [10]:
threshold = movies_df["vote_count"].quantile(0.25) #`.quantile(p)` returns the value below which `p`% of the data falls.
print("25th percentile (vote_count):", threshold)
movies_df.sort_values("vote_count").head(10)

25th percentile (vote_count): 312.5


,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
81,False,/v2SOWxOgifIhnZFLgIEvzr7ZbJ2.jpg,[27],1567187,A Mother's Recall,es,Memoria de una madre,A married couple welcomes young Genaro into th...,53.5837,/vhfrAIIe9jHiuD9qjOhLwnEgLdb.jpg,2025-10-21,False,False,0.0,0
71,False,/ddrLCUT8hqSlEICTTbdlFYB5q4a.jpg,"[28, 35]",1307247,OK! Madam: Bon Voyage,ko,오케이 마담 2,A family fresh off a daring high-altitude plan...,57.5724,/xDbrWXB9yb5aEdt4peVy8OWNP2Y.jpg,2026-08-12,False,False,0.0,0
469,False,/wpx3J1ZuIZgWc9rzdfcj7fDru58.jpg,[27],1400837,Other Mommy,en,Other Mommy,"A young girl, Bela, forms a relationship with ...",17.1104,/8CXwxpyvQiz9COMLZov4Oae8NKb.jpg,2026-10-08,False,False,0.0,0
477,False,/RO1Jus7kHiuFuW2XTzmCVwj6Sr.jpg,[18],1710886,Sugar Mommy,tl,Sugar Mommy,When a non-committal sugar mommy takes in her ...,18.3546,/2FBvobFZGibenqKerk786qg9AHD.jpg,2026-06-09,False,False,0.0,0
442,False,/ks72NqTPhpupk207NXiqmZIQQVI.jpg,"[18, 14, 28]",656908,Ramayana,hi,रामायण,An ancient epic follows a young prince and pri...,17.5165,/f3yZZw7zIsWo6m9xJStfjDauIZX.jpg,2026-11-08,False,False,0.0,0
450,False,None,"[28, 80, 53]",755679,Fast Forever,en,Fast Forever,The eleventh and final installment in The Fast...,19.9442,/xaRzhQZ3zK5lN6vEetG8PD17KPF.jpg,2028-03-16,False,False,0.0,0
431,False,/tmIHXh1VEV20rHvkCixP13J1Unw.jpg,"[10751, 18]",1088428,A Boy and a Girl,zh,少男少女,"In a declining small town, an idle boy encount...",22.4355,/36PslBjdTo0QWrYgX9Xi7AFsIsR.jpg,2023-11-17,False,False,0.0,0
426,False,/1A7s8zG4PF6YoJrncrTO6N4r0Sx.jpg,"[27, 53, 878]",1400940,Clayface,en,Clayface,Follow the terrifying descent into hell of a p...,19.3658,/5jCpQnWPikggmQZoDp1eAi6BI6w.jpg,2026-10-21,False,False,0.0,0
524,False,/cR7shXEoxI2smPD7cZETC9tM9su.jpg,[9648],747845,House of Salt,en,House of Salt,We follow a man who seems to be forever drippi...,15.1716,/iqiqlqAnKBarqlZh82gTJ0lfPe2.jpg,2019-07-10,False,False,0.0,0
343,False,/vO5poa7WpgBqzgiGSVtTdtQrecJ.jpg,[10749],716273,Just A Bite,ko,한입만,Arang doesn't have a quick match with her husb...,23.0019,/fBfbWJiG5CUGr1gjiaVPA6l94RW.jpg,2020-06-11,False,False,0.0,0



Decision: The 25th percentile of `vote_count` was 312.5 votes. I dropped rows below this threshold instead
of picking an arbitrary cutoff to maximize the reliability of the data.

In [11]:
print("Before filtering:", len(movies_df))
movies_df = movies_df[movies_df["vote_count"] > threshold]
print("After filtering:", len(movies_df))

Before filtering: 1000
After filtering: 750


Analizyng bool columns 

In [ ]:
bool_columns = movies_df.select_dtypes(include="bool").columns # `.select_dtypes(include="x")` returns only the columns whose data type is x.
for col in bool_columns:
    print(f"--- {col} ---")
    print(movies_df[col].value_counts())
    print()

--- adult ---
adult
False    750
Name: count, dtype: int64

--- softcore ---
softcore
False    750
Name: count, dtype: int64

--- video ---
video
False    750
Name: count, dtype: int64



Decision: drop boolean columns. `Adult` , `softcore` and `video` showed  no variation ( all False), so they add no useful information.

In [13]:
movies_df = movies_df.drop(columns=bool_columns)
movies_df.columns

Index(['backdrop_path', 'genre_ids', 'id', 'title', 'original_language',
       'original_title', 'overview', 'popularity', 'poster_path',
       'release_date', 'vote_average', 'vote_count'],
      dtype='object')

Decision: drop columns not needed for the research question. `backdrop_path`, `poster_path` are image URLs (not analysable data). `overview` is free text, not needed for a genre/rating/popularity comparison. `original_title` is redundant with `title` for this analysis.


In [14]:
movies_df = movies_df.drop(columns=["backdrop_path", "poster_path", "overview", "original_title"])
movies_df.columns

Index(['genre_ids', 'id', 'title', 'original_language', 'popularity',
       'release_date', 'vote_average', 'vote_count'],
      dtype='object')

Decision: extract release year to compare "over the years". 

In [15]:
movies_df["release_date"] = pd.to_datetime(movies_df["release_date"])
movies_df["release_year"] = movies_df["release_date"].dt.year
movies_df.head()

,genre_ids,id,title,original_language,popularity,release_date,vote_average,vote_count,release_year
0,"[12, 28, 14]",1368337,The Odyssey,en,1012.4511,2026-07-15,8.000,1628,2026
1,"[878, 53]",1275779,Disclosure Day,en,544.2213,2026-06-10,7.392,1896,2026
2,"[28, 14, 878]",454639,Masters of the Universe,en,525.9230,2026-06-03,7.338,1204,2026
4,"[28, 12, 878]",1081003,Supergirl,en,511.2893,2026-06-24,6.600,766,2026
5,"[27, 53]",1339713,Obsession,en,370.8152,2026-05-13,8.300,3811,2026


### Decision: merge the tables genre and movies

`genre_ids` are just numbers, merging with the genre lookup table replaces
them with real genre names, which makes posible grouping them by genre names for future exploration

In [ ]:
movies_exploded = movies_df.explode("genre_ids") #genre_ids is a list per movie, .explode() turns each list item into its own row
movies_with_genres = movies_exploded.merge(
    genres_df,
    left_on="genre_ids",
    right_on="id",
    how="left",
    suffixes=("", "_genre")
)
movies_with_genres.head()

Index(['genre_ids', 'id', 'title', 'original_language', 'popularity',
       'release_date', 'vote_average', 'vote_count', 'release_year',
       'id_genre', 'name'],
      dtype='object')

### Save the prepared data 

In [20]:
movies_df.to_csv("../data/processed/movies.csv", index=False)
genres_df.to_csv("../data/processed/genres.csv", index=False)
movies_with_genres.to_csv("../data/processed/movies_with_genres.csv", index=False)